# 1a · Prompt Test (tiny sample)

Sends **one request** with the **current** prompt of `1_AugmentResponses.ipynb`, for `N_PAIRS` drug pairs. It shows the generated sentences, their labels and every rule they break. Use it to try prompt changes before spending the daily budget on a full run.

- **`TEST_MODE`** chooses the prompt:
  - `"normal"`: three answers per pair (entailment, contradiction, neutral), as for train, val and the normal half of test;
  - `"mixed"`: one mixed-class answer per pair, as for the mixed half of test, with notebook 1's random plan and the atomiser gate.
- **Always the current prompt.** Notebook 1's code (prompts, worked examples, plans, validators) is loaded from that file when this notebook runs. Edit the prompt there and re-run here; there is no copy to keep in sync. To try a variant *without* touching notebook 1, override it in *Try a prompt change*.
- **Fixed, stratified sample from train.** One pair per interaction type, plus a hedged and a negative premise, picked with `SAMPLE_SEED`, so prompt variants are compared on the same pairs. Mixed answers are only generated for test in the real run, but tuning the prompt on train pairs keeps test untouched.
- **One retry round.** With `ROUNDS = 2`, rejected pairs are re-sent once with their rejection reasons, as in the full run.
- **Atomiser check.** If scispaCy is available, every accepted sentence is run through notebook 2's atomiser. A sentence that gives exactly one atom passes its label to that atom unchanged; 0 or 2+ atoms is where label errors and coverage loss start.
- **Nothing in the main data changes.** Results go to `data/prompt_test/`. Requests still count against the shared daily budget (`.gemini_requests.json`).


In [5]:
NB1_PATH = "1_AugmentResponses.ipynb"

TEST_MODE = "normal"         # "normal" (E/C/N responses) or "mixed" (mixed-class answers)
SAMPLE_SPLIT = "train"       # never tune the prompt on test pairs
N_PAIRS = 6                  # one request per round up to PAIRS_PER_REQUEST (normal) / MIXED_ANSWERS_PER_REQUEST (mixed)
SAMPLE_IDS = []              # pin exact pairs by original_id; empty = stratified random sample
SAMPLE_SEED = 7              # change it for a different random sample
ROUNDS = 2                   # 1 = single request; 2 = re-send the rejected pairs once with their reasons
SHOW_PROMPT = True           # print the request prompt (worked examples + item blocks)
SHOW_SYSTEM_PROMPT = False
ATOMISE = True               # run notebook 2's atomiser on the accepted sentences (needs scispaCy)
ATOMISER_FALLBACK = False    # True only if scispaCy is missing (quick look with en_core_web_sm)
OUT_DIR = "data/prompt_test"


## Load notebook 1 (and notebook 2's atomiser)

In [6]:
import json, os, time
from datetime import datetime
from IPython.display import display


def load_notebook_code(path, namespace, stop_marker):
    """Runs the code cells of `path` in `namespace`, in order, up to the first cell that contains `stop_marker`.
    Lines starting with % or ! (pip installs) are skipped. An error names the cell it came from
    ('<1_AugmentResponses.ipynb, code cell 7>') and shows its code."""
    import linecache
    with open(path, encoding="utf-8") as f:
        cells = [c for c in json.load(f)["cells"] if c["cell_type"] == "code"]
    for number, cell in enumerate(cells, 1):
        src = "".join(cell["source"])
        if stop_marker in src:
            return
        src = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("%", "!")))
        name = f"<{os.path.basename(path)}, code cell {number}>"
        linecache.cache[name] = (len(src), None, src.splitlines(True), name)
        exec(compile(src, name, "exec"), namespace)
    raise ValueError(f"{path}: no cell contains {stop_marker!r}")


# Notebook 1: everything before the generation run (config, client, pair selection, plans, prompts, validators,
# request helpers). Loaded into this notebook's namespace, so the overrides below take effect.
load_notebook_code(NB1_PATH, globals(), "await generate_split(")
assert TEST_MODE in ("normal", "mixed"), TEST_MODE
os.makedirs(OUT_DIR, exist_ok=True)

# Notebook 2's atomiser, through notebook 1's loader (its own namespace, loaded once).
ATOMISER_ALLOW_FALLBACK = ATOMISER_FALLBACK
if ATOMISE or (TEST_MODE == "mixed" and MIXED_ATOM_GATE):
    if get_atomiser(required=False) is None and TEST_MODE == "mixed":
        MIXED_ATOM_GATE = False
        print("→ the mixed answers are checked WITHOUT the atomiser gate in this test")
print(f"\nLoaded {NB1_PATH}: mode {TEST_MODE}, model {MODEL_NAME}; atomiser {'on' if ATOMISER else 'off'}"
      + (f"; gate {'on' if MIXED_ATOM_GATE else 'off'}" if TEST_MODE == "mixed" else ""))


c:\Users\kynes\thesiskernel\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Corpus drug names: 2,372; 41 of them mostly ordinary words, not counted as drugs (adrenergic, alkalinizing agents, analgesics, anticholinergic, anticholinesterase, antiretroviral, azoles, bismuth, ...); brand/generic pairs named in brackets: 63
8 worked examples pass the validator.
mixed worked example passes the validator.


c:\Users\kynes\thesiskernel\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Loaded scispaCy en_core_sci_sm (biomedical parser).
pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']
Drug vocabulary: 2,415 names annotated in train_raw_manifest.csv, val_raw_manifest.csv, test_raw_manifest.csv; 43 mostly ordinary words are only taken as the labelled drugs (adrenergic, alkalinizing agents, analgesics, anticholinergic, anticholinesterase, antiretroviral, azoles, bismuth, ...)
Lexicon: 141 words, of which 20 were added after reading generated text.
Extraction helpers defined.
atomize_dataframe() and report_coverage() defined.

Loaded 1_AugmentResponses.ipynb: mode normal, model gemini-3.5-flash-lite; atomiser on


## Try a prompt change (optional)

Anything set in this cell overrides notebook 1 **for this test only**: the request functions read these names when they run. Leave it empty to test notebook 1 as it is. When a variant works, copy it into notebook 1.

In [7]:
# Examples (uncomment to try):
# SYSTEM_PROMPT = SYSTEM_PROMPT.replace("40-80 words in total", "45-75 words in total")      # normal (+ shared rules)
# MIXED_RULES = MIXED_RULES.replace("keep it simple", "keep it short and simple")            # mixed only
# WORKED_EXAMPLES = WORKED_EXAMPLES[:4]
# LABEL_WEIGHTS = {"entailment": 2.0, "contradiction": 1.0, "neutral": 1.0}                  # mixed compositions
# MODEL_NAME, TEMPERATURE, THINKING_BUDGET = "gemini-3.5-flash-lite", 0.4, 1024

# Changed worked examples must still follow the rules they teach:
for ex in WORKED_EXAMPLES:
    result, problems = check_item(json.loads(json.dumps(ex["output"])), example_item(ex))
    if result is None or problems:
        print(f"worked example {ex['drug1']} / {ex['drug2']}: {problems}")
ok, hard, soft = check_mixed(json.loads(json.dumps(MIXED_EXAMPLE["sentences"])),
                             {"pair_id": "example", "premise": MIXED_EXAMPLE["premise"], "drug1": MIXED_EXAMPLE["drug1"],
                              "drug2": MIXED_EXAMPLE["drug2"], "plan": example_plan(MIXED_EXAMPLE)}, gate=False)
if ok is None or soft:
    print(f"mixed worked example: {hard + soft}")
if SHOW_SYSTEM_PROMPT:
    print(SYSTEM_PROMPT if TEST_MODE == "normal" else mixed_system_prompt())


## Sample

In [8]:
pairs, references = prepare_pairs(SAMPLE_SPLIT)

if SAMPLE_IDS:
    sample = pairs[pairs["original_id"].isin(SAMPLE_IDS)]
    missing = set(SAMPLE_IDS) - set(sample["original_id"])
    if missing:
        print(f"not in the pair selection of {SAMPLE_SPLIT}: {sorted(missing)}")
else:
    rng = random.Random(SAMPLE_SEED)
    body = pairs["premise"].str.replace(TAG_RE, "", regex=True)
    picks = []

    def take(mask):
        candidates = [i for i in pairs.index[mask] if i not in picks]
        if candidates and len(picks) < N_PAIRS:
            picks.append(rng.choice(candidates))

    for interaction_type in sorted(pairs["interaction_type"].dropna().unique()):
        take(pairs["interaction_type"] == interaction_type)
    take(body.str.contains(HEDGE_RE))
    take(body.str.contains(NEGATION_RE))
    while len(picks) < min(N_PAIRS, len(pairs)):
        take(pd.Series(True, index=pairs.index))
    sample = pairs.loc[picks]

if TEST_MODE == "normal":
    items = [{**item_base(r), "pair_id": r["original_id"], "plan": make_plan(r, references)} for _, r in sample.iterrows()]
    describe = lambda it: {"contradiction": " then ".join(it["plan"].contradiction_strategies),
                           "neutral": " then ".join(it["plan"].neutral_strategies)}
else:
    items = [{**item_base(r), "pair_id": f"{r['original_id']}#m0", "plan": make_mixed_plan(r, references, 0)}
             for _, r in sample.iterrows()]
    describe = lambda it: {"plan": " ".join(LETTER[l] for l in it["plan"].labels),
                           "parts": "; ".join(f"{k} '{w}'" for k, w in it["plan"].parts)}

pd.set_option("display.max_colwidth", 160)
display(pd.DataFrame([{"pair": it["pair_id"], "type": it["interaction_type"], "drugs": f"{it['drug1']} / {it['drug2']}",
                       "hedged": it["plan"].premise_hedged, "negative": it["plan"].premise_negative, **describe(it),
                       "premise": it["premise"]} for it in items]))


[train] 6,898 pairs → −165 same substance, −161 heading premise, −981 over 4/premise → 5,600


,pair,type,drugs,hedged,negative,contradiction,neutral,premise
0,DDI-DrugBank.d115.s0.p14,advise,Gleevec / clarithromycin,True,False,deny then conflicting_fact,borrowed_effect then added_detail,[DRUGBANK] Drugs that may alter imatinib plasma concentrations Drugs that may increase imatinib plasma concentrations: Caution is recommended when administe...
1,DDI-DrugBank.d405.s1.p4,effect,barbiturates / ethchlorvynol,True,False,deny then conflicting_fact,borrowed_effect then added_detail,"[DRUGBANK] Other depressasnts such as alcohol, barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol."
2,DDI-DrugBank.d441.s1.p0,int,linezolid / adrenergic,True,False,deny then deny,borrowed_effect then added_detail,"[DRUGBANK] Therefore, linezolid has the potential for interaction with adrenergic and serotonergic agents."
3,DDI-MedLine.d111.s12.p0,mechanism,Valproic acid / phenobarbital,False,False,reverse_direction then deny,borrowed_effect then added_detail,"[MEDLINE] Valproic acid diminished binding of phenobarbital by a net change of 9.9% (percentage increase in FDF, 21.2%) at 1732 micromol/L."
4,DDI-MedLine.d83.s3.p0,none,olanzapine / haloperidol,False,False,conflicting_fact then deny,borrowed_effect then added_detail,"[MEDLINE] In large, well controlled trials in patients with schizophrenia or related psychoses, olanzapine 5 to 20 mg/day was significantly superior to halo..."
5,DDI-DrugBank.d409.s0.p0,advise,tetracyclines / anticoagulant,True,False,deny then conflicting_fact,borrowed_effect then added_detail,"[DRUGBANK] Because the tetracyclines have been shown to depress plasma prothrombin activity, patients who are on anticoagulant therapy may require downward ..."


## Run

One request per round (per `PAIRS_PER_REQUEST` pairs in normal mode, `MIXED_ANSWERS_PER_REQUEST` in mixed mode). Every problem is collected, not only the first three that are sent back as feedback.

In [9]:
def call_model(prompt, config_fn):
    """One request, counted against the daily budget. Returns the response, or None when the budget is used up."""
    errors = 0
    while True:
        if not budget.take():
            print(f"Daily request budget used up ({budget.sent}/{DAILY_REQUEST_BUDGET}).")
            return None
        try:
            return client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config_fn())
        except Exception as e:
            if _thinking["enabled"] and "thinking" in str(e).lower():
                budget.refund()
                _thinking["enabled"] = False
                print(f"{MODEL_NAME} does not accept a thinking budget; continuing without it.")
                continue
            is_daily, retry_delay = classify_rate_limit(e)
            if is_daily:
                budget.exhaust()
                print("Daily quota reached (re-run after the reset, 14:00 WIB).")
                return None
            if retry_delay is not None:
                budget.refund()
                time.sleep(retry_delay + 1)
                continue
            errors += 1
            if errors >= TRANSPORT_RETRIES:
                raise
            time.sleep(min(2.0 ** errors, 60.0))


def check_all(text, chunk):
    """{pair_id: result} with every hard and soft problem (the full run only sends back the first three)."""
    try:
        data = json.loads((text or "").strip())
    except json.JSONDecodeError:
        data = None
    if not isinstance(data, list):
        return {it["pair_id"]: {"accepted": False, "hard": ["the response was not a JSON array"], "soft": [],
                                "answer": None} for it in chunk}
    by_item_id = {f"P{i + 1}": it for i, it in enumerate(chunk)}
    results = {}
    for entry in data:
        item = by_item_id.get(str(entry.get("item_id", "")).strip()) if isinstance(entry, dict) else None
        if item is None or item["pair_id"] in results:
            continue
        if TEST_MODE == "mixed":
            sentences = entry.get("sentences")
            accepted, hard, soft = check_mixed(sentences, item)
            results[item["pair_id"]] = {"accepted": accepted is not None, "hard": hard, "soft": soft,
                                        "answer": sentences if isinstance(sentences, list) else None}
            continue
        responses = entry.get("responses")
        error = structure_error(responses)
        by_label = ({r.get("label"): r for r in responses if isinstance(r, dict)}
                    if isinstance(responses, list) else None)
        if error:
            results[item["pair_id"]] = {"accepted": False, "hard": [error], "soft": [], "answer": by_label}
            continue
        hard = hard_problems(by_label, item)
        results[item["pair_id"]] = {"accepted": not hard, "hard": hard, "soft": soft_problems(by_label, item),
                                    "answer": by_label}
    for it in chunk:
        results.setdefault(it["pair_id"], {"accepted": False, "hard": ["the item was missing from the response"],
                                           "soft": [], "answer": None})
    return results


spec = MODES[TEST_MODE]
results, raw, log = {}, [], []
queue = list(items)
for round_no in range(1, ROUNDS + 1):
    if not queue:
        break
    retry, per_request = [], spec["per_request"]()
    for start in range(0, len(queue), per_request):
        chunk = queue[start:start + per_request]
        prompt = spec["prompt"]([{**it, "item_id": f"P{i + 1}"} for i, it in enumerate(chunk)])
        if SHOW_PROMPT and round_no == 1 and start == 0:
            print(prompt, "\n" + "─" * 100)
        t0 = time.time()
        response = call_model(prompt, lambda: model_config(spec["system"]()))
        if response is None:
            retry = []
            break
        usage = getattr(response, "usage_metadata", None)
        log.append({"round": round_no, "items": len(chunk), "seconds": round(time.time() - t0, 1),
                    "prompt_tokens": getattr(usage, "prompt_token_count", None),
                    "output_tokens": getattr(usage, "candidates_token_count", None),
                    "thinking_tokens": getattr(usage, "thoughts_token_count", None)})
        raw.append({"round": round_no, "items": [it["pair_id"] for it in chunk], "prompt": prompt,
                    "response": response.text})
        checked = check_all(response.text, chunk)
        for item in chunk:
            res = checked[item["pair_id"]]
            res["round"] = round_no
            results[item["pair_id"]] = res
            if not res["accepted"]:
                retry.append({**item, "attempts": item["attempts"] + 1, "feedback": "; ".join(res["hard"][:3])})
    queue = retry

display(pd.DataFrame(log))
print(f"requests used today: {budget.sent}/{DAILY_REQUEST_BUDGET}")


WORKED EXAMPLES

Example 1 (different drugs; do NOT reuse its wording)
Drug 1: digoxin
Drug 2: quinidine
Interaction type: advise
Premise: Dose reduction of digoxin is recommended when quinidine is started.
Premise certainty: definite
Premise is negative: no
Plan: entailment=paraphrase; contradiction=opposite_advice then reverse_direction (premise says 'reduction' -> increase / raise); neutral=borrowed_effect (reference: '[drug A] increases the risk of bleeding in patients taking [drug B].') then added_detail
Sentence labels: entailment response = entailment, entailment, entailment, entailment; contradiction response = contradiction, entailment, entailment, contradiction; neutral response = neutral, entailment, neutral, entailment
Output responses: [{"label": "entailment", "sentences": [{"label": "entailment", "text": "Starting quinidine calls for a lower dose of digoxin.", "evidence": "Dose reduction of digoxin is recommended when quinidine is started"}, {"label": "entailment", "text"

,round,items,seconds,prompt_tokens,output_tokens,thinking_tokens
0,1,6,22.3,5116,5116,5332
1,2,6,26.9,5451,5057,7381


requests used today: 2/500


## Report

Per pair: the plan, then every sentence with its label, word count, evidence check and the evidence or "missing" text. In mixed mode, each sentence also shows its planned target. After that come every hard problem (the reason for rejection) and every soft problem (logged only).

In [10]:
def sentence_rows(answer, item):
    """[(response, sentences)] for the table: three responses (normal) or one answer (mixed)."""
    if TEST_MODE == "mixed":
        return [("answer", answer or [])]
    return [(rlabel, ((answer or {}).get(rlabel) or {}).get("sentences") or []) for rlabel in LABELS]


def sentence_table(answer, item):
    rows = []
    targets = item["plan"].targets if TEST_MODE == "mixed" else []
    for rlabel, sentences in sentence_rows(answer, item):
        for k, s in enumerate(sentences, start=1):
            if not isinstance(s, dict):
                continue
            label, evidence = s.get("label"), s.get("evidence", "")
            rows.append({
                "response": rlabel, "#": k, "label": label, "text": s.get("text", ""),
                **({"planned": targets[k - 1] if k <= len(targets) else ""} if TEST_MODE == "mixed" else {}),
                "words": word_count(s.get("text", "")),
                "evidence ok": ("✓" if is_quote(evidence, item["premise"]) else "✗")
                               if label in ("entailment", "contradiction") else "",
                "evidence / missing": evidence or s.get("missing", ""),
            })
    return pd.DataFrame(rows)


for item in items:
    res = results.get(item["pair_id"])
    print("=" * 110)
    print(f"{item['pair_id']}  [{item['interaction_type']}]  {item['drug1']} / {item['drug2']}")
    print(f"Premise: {item['premise']}")
    print("Plan:", describe(item), f"hedged={item['plan'].premise_hedged}, negative={item['plan'].premise_negative}")
    if res is None:
        print("not sent (budget)")
        continue
    print(f"→ {'ACCEPTED' if res['accepted'] else 'REJECTED'} in round {res['round']}")
    table = sentence_table(res["answer"], item)
    if len(table):
        display(table)
        print("words:", table.groupby("response")["words"].sum().to_dict())
    for p in res["hard"]:
        print("  HARD:", p)
    for p in res["soft"]:
        print("  soft:", p)

accepted = [it for it in items if results.get(it["pair_id"], {}).get("accepted")]
first_round = sum(1 for it in accepted if results[it["pair_id"]]["round"] == 1)
print("=" * 110)
print(f"accepted: {len(accepted)}/{len(items)} ({first_round} in round 1)")
reasons = Counter(reason_key(p) for r in results.values() for p in r["hard"])
if reasons:
    print("hard problems (all rounds):")
    for reason, n in reasons.most_common():
        print(f"  {n:3d}  {reason}")


def output_rows(item):
    res = results[item["pair_id"]]
    if TEST_MODE == "mixed":
        return mixed_to_rows(item, res["answer"], res["soft"])
    return to_rows(item["row"], item["plan"], [res["answer"][l] for l in LABELS], res["soft"])


DDI-DrugBank.d115.s0.p14  [advise]  Gleevec / clarithromycin
Premise: [DRUGBANK] Drugs that may alter imatinib plasma concentrations Drugs that may increase imatinib plasma concentrations: Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin).
Plan: {'contradiction': 'deny then conflicting_fact', 'neutral': 'borrowed_effect then added_detail'} hedged=True, negative=False
→ REJECTED in round 2


,response,#,label,text,words,evidence ok,evidence / missing
0,entailment,1,entailment,"When clarithromycin is given with Gleevec, caution is recommended.",9,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"
1,entailment,2,entailment,Providers should exercise caution when administering clarithromycin and Gleevec together.,10,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"
2,entailment,3,entailment,Careful monitoring is advised upon administering Gleevec and clarithromycin concurrently.,10,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"
3,entailment,4,entailment,Prescribers may need to take caution when Gleevec is administered alongside clarithromycin.,12,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"
4,contradiction,1,entailment,Caution is recommended when administering Gleevec and clarithromycin together.,9,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"
5,contradiction,2,entailment,Prescribing Gleevec with clarithromycin may require caution.,7,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"
6,contradiction,3,contradiction,The administration of Gleevec together with clarithromycin fails to modify imatinib plasma concentrations.,13,✓,Drugs that may increase imatinib plasma concentrations
7,contradiction,4,contradiction,Administration of Gleevec and clarithromycin requires no caution.,8,✓,Caution is recommended when administering Gleevec
8,neutral,1,neutral,Combinations of Gleevec and clarithromycin were indifferent for 29 strains and synergistic for 33 strains.,15,,microbial strain sensitivity
9,neutral,2,entailment,Caution is recommended when administering Gleevec with clarithromycin.,8,✓,"Caution is recommended when administering Gleevec with inhibitors of the CYP3A4 family (e.g., ketoconazole, itraconazole, erythromycin, clarithromycin)"


words: {'contradiction': 37, 'entailment': 41, 'neutral': 42}
  HARD: contradiction response has 37 words (needs 40-80)
  soft: neutral#1 (neutral): hedge missing
  soft: neutral#4 (neutral): hedge missing
DDI-DrugBank.d405.s1.p4  [effect]  barbiturates / ethchlorvynol
Premise: [DRUGBANK] Other depressasnts such as alcohol, barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol.
Plan: {'contradiction': 'deny then conflicting_fact', 'neutral': 'borrowed_effect then added_detail'} hedged=True, negative=False
→ REJECTED in round 2


,response,#,label,text,words,evidence ok,evidence / missing
0,entailment,1,entailment,Barbiturates may increase CNS depression when given with ethchlorvynol.,9,✓,"barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol"
1,entailment,2,entailment,CNS depression could be enhanced when barbiturates are administered with ethchlorvynol.,11,✓,"barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol"
2,entailment,3,entailment,"When barbiturates are administered with ethchlorvynol, CNS depression might be enhanced.",11,✓,"barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol"
3,entailment,4,entailment,Ethchlorvynol given with barbiturates may heighten CNS depression.,8,✓,"barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol"
4,contradiction,1,entailment,Barbiturates may enhance CNS depression when administered with ethchlorvynol.,9,✓,"barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol"
5,contradiction,2,entailment,Ethchlorvynol taken with barbiturates could enhance CNS depression.,8,✓,"barbiturates, and MAOIs may enhance CNS depression when administered with ethchlorvynol"
6,contradiction,3,contradiction,Barbiturates shows no difference in CNS depression when administered with ethchlorvynol.,11,✓,may enhance CNS depression
7,contradiction,4,contradiction,Barbiturates decrease CNS depression when administered with ethchlorvynol.,8,✓,may enhance CNS depression
8,neutral,1,neutral,"Ethchlorvynol may antagonize the effects of the drugs that govern gastrointestinal motility, such as barbiturates.",15,,gastrointestinal motility effects
9,neutral,2,neutral,Barbiturates and ethchlorvynol enhance CNS depression for a duration of four hours.,12,,the exact duration of the effect


words: {'contradiction': 36, 'entailment': 39, 'neutral': 45}
  HARD: entailment response has 39 words (needs 40-80)
  HARD: contradiction response has 36 words (needs 40-80)
  soft: neutral#2 (neutral): hedge missing
  soft: neutral: entailment sentences repeat each other
DDI-DrugBank.d441.s1.p0  [int]  linezolid / adrenergic
Premise: [DRUGBANK] Therefore, linezolid has the potential for interaction with adrenergic and serotonergic agents.
Plan: {'contradiction': 'deny then deny', 'neutral': 'borrowed_effect then added_detail'} hedged=True, negative=False
→ REJECTED in round 2


,response,#,label,text,words,evidence ok,evidence / missing
0,entailment,1,entailment,Linezolid might interact with adrenergic agents.,6,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents
1,entailment,2,entailment,Adrenergic agents could interact with linezolid.,6,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents
2,entailment,3,entailment,A possible pharmacologic interaction exists between linezolid and adrenergic agents.,10,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents
3,entailment,4,entailment,Linezolid may affect adrenergic agents.,5,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents
4,contradiction,1,contradiction,Linezolid is unaffected by adrenergic agents.,6,✓,linezolid has the potential for interaction with adrenergic
5,contradiction,2,contradiction,Linezolid does not change when combined with adrenergic agents.,9,✓,potential for interaction with adrenergic
6,contradiction,3,entailment,Linezolid has the potential for interaction with adrenergic agents.,9,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents
7,contradiction,4,entailment,Adrenergic agents may interact with linezolid.,6,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents
8,neutral,1,neutral,"Based on the chemical resemblance of linezolid and adrenergic agents, coadministration of linezolid with adrenergic agents is contraindicated.",18,,contraindication status based on chemical resemblance
9,neutral,2,entailment,Linezolid has the potential for interaction with adrenergic agents.,9,✓,linezolid has the potential for interaction with adrenergic and serotonergic agents


... (3 outputs left out)

,response,#,label,text,words,evidence ok,evidence / missing
0,entailment,1,entailment,Olanzapine was superior to haloperidol in overall improvements in psychopathology rating scales.,12,✓,olanzapine 5 to 20 mg/day was significantly superior to haloperidol 5 to 20 mg/day in overall improvements in psychopathology rating scales
1,entailment,2,entailment,"In psychopathology rating scales, olanzapine showed superiority over haloperidol.",9,✓,olanzapine 5 to 20 mg/day was significantly superior to haloperidol 5 to 20 mg/day in overall improvements in psychopathology rating scales
2,entailment,3,entailment,Olanzapine and haloperidol had comparable effects on positive psychotic symptoms.,10,✓,was comparable in effects on positive psychotic symptoms
3,entailment,4,entailment,The effects of olanzapine and haloperidol on positive psychotic symptoms were comparable.,12,✓,was comparable in effects on positive psychotic symptoms
4,contradiction,1,entailment,Olanzapine was superior to haloperidol in overall improvements in psychopathology rating scales.,12,✓,olanzapine 5 to 20 mg/day was significantly superior to haloperidol 5 to 20 mg/day in overall improvements in psychopathology rating scales
5,contradiction,2,contradiction,Haloperidol was superior to olanzapine in overall improvements in psychopathology rating scales.,12,✓,olanzapine 5 to 20 mg/day was significantly superior to haloperidol 5 to 20 mg/day in overall improvements in psychopathology rating scales
6,contradiction,3,entailment,Olanzapine and haloperidol had comparable effects on positive psychotic symptoms.,10,✓,was comparable in effects on positive psychotic symptoms
7,contradiction,4,contradiction,Olanzapine has no effect on positive psychotic symptoms compared to haloperidol.,11,✓,was comparable in effects on positive psychotic symptoms
8,neutral,1,entailment,Olanzapine was superior to haloperidol in overall improvements in psychopathology rating scales.,12,✓,olanzapine 5 to 20 mg/day was significantly superior to haloperidol 5 to 20 mg/day in overall improvements in psychopathology rating scales
9,neutral,2,neutral,Elevated olanzapine levels have been reported in postmarketing experience when haloperidol is administered concomitantly.,14,,postmarketing elevated levels


words: {'contradiction': 45, 'entailment': 43, 'neutral': 52}
  HARD: contradiction response, sentence 2 (contradiction): a reordering of an entailment sentence (role swap)
  HARD: contradiction response, sentence 2 (contradiction): repeats the premise
  soft: entailment: entailment sentences repeat each other
DDI-DrugBank.d409.s0.p0  [advise]  tetracyclines / anticoagulant
Premise: [DRUGBANK] Because the tetracyclines have been shown to depress plasma prothrombin activity, patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage.
Plan: {'contradiction': 'deny then conflicting_fact', 'neutral': 'borrowed_effect then added_detail'} hedged=True, negative=False
→ ACCEPTED in round 2


,response,#,label,text,words,evidence ok,evidence / missing
0,entailment,1,entailment,Patients on anticoagulant therapy may require downward adjustment of their anticoagulant dosage when taking tetracyclines.,15,✓,patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage
1,entailment,2,entailment,A downward adjustment of anticoagulant dosage may be required for patients on anticoagulant therapy when taking tetracyclines.,17,✓,patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage
2,entailment,3,entailment,"Tetracyclines depress plasma prothrombin activity, which may lead to a downward adjustment of anticoagulant dosage.",15,✓,the tetracyclines have been shown to depress plasma prothrombin activity
3,entailment,4,entailment,Anticoagulant dosage may need a downward adjustment in patients receiving tetracyclines.,11,✓,patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage
4,contradiction,1,entailment,Patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage when taking tetracyclines.,17,✓,patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage
5,contradiction,2,entailment,Tetracyclines have been shown to depress plasma prothrombin activity in patients taking anticoagulant.,13,✓,the tetracyclines have been shown to depress plasma prothrombin activity
6,contradiction,3,contradiction,Anticoagulant therapy shows no difference in dosage adjustment when combined with tetracyclines.,12,✓,may require downward adjustment of their anticoagulant dosage
7,contradiction,4,contradiction,Tetracyclines increase plasma prothrombin activity during anticoagulant therapy.,8,✓,the tetracyclines have been shown to depress plasma prothrombin activity
8,neutral,1,entailment,Patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage when taking tetracyclines.,17,✓,patients who are on anticoagulant therapy may require downward adjustment of their anticoagulant dosage
9,neutral,2,entailment,Tetracyclines have been shown to depress plasma prothrombin activity in patients taking anticoagulant.,13,✓,the tetracyclines have been shown to depress plasma prothrombin activity


words: {'contradiction': 50, 'entailment': 58, 'neutral': 54}
  soft: neutral#3 (neutral): hedge missing
  soft: neutral#4 (neutral): hedge missing
accepted: 1/6 (0 in round 1)
hard problems (all rounds):
    4  contradiction response has # words …
    3  entailment response has # words …
    1  contradiction response, sentence # …: a reordering of an entailment sentence …
    1  contradiction response, sentence # …: repeats the premise


## Atomiser check

Runs notebook 2's atomiser on the accepted answers. Each sentence should give **exactly one** atom:
- 0 atoms means the claim is lost for the atomic models (coverage);
- 2+ atoms means one sentence label is spread over several claims, which may not all deserve it.

In mixed mode the gate already enforced this when it was on.

In [11]:
if ATOMISER is None:
    print("Atomiser not loaded (see the loading cell).")
elif not accepted:
    print("No accepted answers to atomise.")
else:
    holistic = pd.DataFrame([row for it in accepted for row in output_rows(it)])
    holistic.insert(0, "source_row", range(len(holistic)))
    atoms = ATOMISER["atomize_dataframe"](holistic, split_name="prompt test")
    ATOMISER["report_coverage"](atoms, "prompt test")

    found = atoms[atoms["source_tier"] != "filtered_out"]
    per_sentence = []
    for _, r in holistic.iterrows():
        for k, claim in enumerate(json.loads(r["claims"])):
            hits = found[(found["source_row"] == r["source_row"]) & (found["claim_idx"] == k)]
            per_sentence.append({"item": r["item_id"], "answer label": r["label"], "#": k + 1, "label": claim["label"],
                                 "atoms": len(hits), "sentence": claim["text"],
                                 "atom text": " | ".join(hits["hypothesis"])})
    per_sentence = pd.DataFrame(per_sentence)
    bucket = per_sentence["atoms"].clip(upper=2).map({0: "0 atoms", 1: "1 atom", 2: "2+ atoms"})
    print("sentences by number of atoms, per sentence label:")
    display(pd.crosstab(per_sentence["label"], bucket, margins=True))
    off = per_sentence[per_sentence["atoms"] != 1]
    if len(off):
        print("sentences that do not give exactly one atom:")
        display(off)


[prompt test] responses with >=1 atom: 3/3 (100.0%)
  by response label : contradiction 100.0%, entailment 100.0%, neutral 100.0%
  claims with >=1 atom: 11/12 (91.7%)
  by claim label    : contradiction 100.0%, entailment 87.5%, neutral 100.0%
  atoms: 11  (avg 3.67 per covered response)
  atom labels       : {'entailment': 7, 'contradiction': 2, 'neutral': 2}
  from split conjuncts: 0 | repaired with whole clause: 0 | context trimmed: 0 | roles by text order: 5
  no atom, by reason: {'no_atom': 1}

sentences by number of atoms, per sentence label:


atoms,0 atoms,1 atom,All
label,,,
contradiction,0,2,2
entailment,1,7,8
neutral,0,2,2
All,1,11,12


sentences that do not give exactly one atom:


,item,answer label,#,label,atoms,sentence,atom text
2,DDI-DrugBank.d409.s0.p0,entailment,3,entailment,0,"Tetracyclines depress plasma prothrombin activity, which may lead to a downward adjustment of anticoagulant dosage.",


## Save

In [12]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
rows = [row for it in accepted for row in output_rows(it)]
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/prompt_test_{TEST_MODE}_{stamp}.csv", index=False)
with open(f"{OUT_DIR}/prompt_test_{TEST_MODE}_{stamp}.json", "w", encoding="utf-8") as f:
    json.dump({"mode": TEST_MODE, "model": MODEL_NAME, "temperature": TEMPERATURE, "split": SAMPLE_SPLIT, "log": log,
               "problems": {pid: {"accepted": r["accepted"], "round": r["round"], "hard": r["hard"], "soft": r["soft"]}
                            for pid, r in results.items()},
               "requests": raw}, f, ensure_ascii=False, indent=1)
print(f"saved {OUT_DIR}/prompt_test_{TEST_MODE}_{stamp}.csv ({len(rows)} answers) and the raw prompts/responses (.json)")


saved data/prompt_test/prompt_test_normal_20261028_153352.csv (3 answers) and the raw prompts/responses (.json)
